<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/Window_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ts2vg
!pip install mne
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp

In [ ]:
# extract a window of exactly `window_size_s` seconds from the DataFrame
# if df cointains a seizure: window contains only ictal timepoints, if no seizure: random position
def extract_window(df, window_size_s=10, mode="ictal"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx : idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    start_idx = int(np.random.choice(valid_starts))
    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window

In [ ]:
# loop aover all .seizures files, process each edf and extract labelled windows
# returns: all_dataframes (dict of full labelled df's, windows (dict of extracted windows)
def process_all_files(pattern="*.seizures", window_size_s=10):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}

    all_dataframes = {}
    windows = {}

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # 1. full dataframe
            df, has_seizure = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # 2. ictal / interictal window
            ictal_window = extract_window(df, window_size_s=window_size_s, mode="ictal")
            interictal_window = extract_window(df, window_size_s=window_size_s, mode="interictal")

            # 3. windows dict
            ictal_name = f"window_ictal_{i}"
            interictal_name = f"window_interictal_{i}"

            windows[ictal_name] = ictal_window
            windows[interictal_name] = interictal_window

            # 4. summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure          : {has_seizure}")

            print(f"  Ictal window name    : {ictal_name}")
            print(f"  Ictal window size    : {len(ictal_window)} samples ({window_size_s}s)")
            print(f"  Ictal time range     : {ictal_window.index[0]:.1f}s → {ictal_window.index[-1]:.1f}s")
            print(f"  Ictal labels         : {ictal_window['label'].value_counts().to_dict()}")

            print(f"  Interictal name      : {interictal_name}")
            print(f"  Interictal size      : {len(interictal_window)} samples ({window_size_s}s)")
            print(f"  Interictal time range: {interictal_window.index[0]:.1f}s → {interictal_window.index[-1]:.1f}s")
            print(f"  Interictal labels    : {interictal_window['label'].value_counts().to_dict()}")

        except FileNotFoundError:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")

    return all_dataframes, windows

In [ ]:
# Run it
all_dataframes, windows = process_all_files(str(RAW_DIR / "*.seizures"), window_size_s=5)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)